In [1]:
import numpy as np
import unittest

def run_tests(*cases):
    suite = unittest.TestSuite()
    for c in cases:
        suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(c))
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    assert result.wasSuccessful(), "Some tests failed"

# ---- GPT-2 small hyperparameters (Section 2) ----
HP = dict(
    l_max=1024,      # max sequence length
    d_e=768,         # model width
    L=12,            # layers
    d_mlp=3072,      # MLP hidden width
    H=12,            # heads
    d_attn=64,       # query/key dim per head
    d_mid=64,        # value dim per head (Alg. 5 name)
    N_V=50257,       # vocabulary
)

def attention_params_per_layer(H, d_attn, d_mid, d_x, d_z, d_out):
    per_head = (d_attn*d_x + d_attn) + (d_attn*d_z + d_attn) + (d_mid*d_z + d_mid)
    return H*per_head + (d_out*H*d_mid + d_out)

def gpt2_param_count(hp, tied=True):
    d_e, d_mlp, L, V, lmax = hp['d_e'], hp['d_mlp'], hp['L'], hp['N_V'], hp['l_max']
    attn = attention_params_per_layer(hp['H'], hp['d_attn'], hp['d_mid'], d_e, d_e, d_e)
    mlp = (d_mlp*d_e + d_mlp) + (d_e*d_mlp + d_e)
    ln = 4*d_e
    total = d_e*V + d_e*lmax + L*(attn + ln + mlp) + 2*d_e
    if not tied:
        total += d_e*V
    return total, attn, mlp

class TestHyperparameters(unittest.TestCase):
    def test_head_dims(self):
        self.assertEqual(HP['d_attn'], HP['d_e'] // HP['H'])
        self.assertEqual(HP['H'] * HP['d_mid'], HP['d_e'])
    def test_attention_params_per_layer(self):
        _, attn, _ = gpt2_param_count(HP)
        self.assertEqual(attn, 2_362_368)
    def test_mlp_params_per_layer(self):
        _, _, mlp = gpt2_param_count(HP)
        self.assertEqual(mlp, 4_722_432)
    def test_total_params_tied(self):
        self.assertEqual(gpt2_param_count(HP)[0], 124_439_808)
    def test_total_params_untied(self):
        total = gpt2_param_count(HP, tied=False)[0]
        self.assertAlmostEqual(total/1e6, 163.0, delta=0.1)

run_tests(TestHyperparameters)

test_attention_params_per_layer (__main__.TestHyperparameters.test_attention_params_per_layer) ... 

ok


test_head_dims (__main__.TestHyperparameters.test_head_dims) ... 

ok


test_mlp_params_per_layer (__main__.TestHyperparameters.test_mlp_params_per_layer) ... 

ok


test_total_params_tied (__main__.TestHyperparameters.test_total_params_tied) ... 

ok


test_total_params_untied (__main__.TestHyperparameters.test_total_params_untied) ... 

ok


----------------------------------------------------------------------
Ran 5 tests in 0.012s

OK


## Algorithm 1: Token embedding
`e = W_e[:, v]`, where `W_e ∈ ℝ^{d_e × N_V}`. A vectorised form accepts an array of IDs and returns a `d_e × l` matrix (the columns `e_1 … e_l`).

In [2]:
def token_embedding(v, W_e):
    '''Algorithm 1. v: int (or int array of length l), 0-indexed token id(s).
    W_e: (d_e, N_V). Returns e of shape (d_e,) for an int, or (d_e, l) for an array.'''
    v = np.asarray(v)
    if not np.issubdtype(v.dtype, np.integer):
        raise TypeError("token ids must be integers")
    if v.min() < 0 or v.max() >= W_e.shape[1]:
        raise IndexError("token id out of vocabulary range")
    return W_e[:, v]

In [3]:
class TestTokenEmbedding(unittest.TestCase):
    def setUp(self):
        rng = np.random.default_rng(0)
        self.d_e, self.N_V = 768, 50257
        self.W_e = rng.standard_normal((self.d_e, self.N_V)).astype(np.float32)

    def test_single_token_shape_and_value(self):
        e = token_embedding(123, self.W_e)
        self.assertEqual(e.shape, (self.d_e,))
        np.testing.assert_array_equal(e, self.W_e[:, 123])

    def test_sequence_shape(self):
        ids = np.array([5, 0, 50256, 5])
        E = token_embedding(ids, self.W_e)
        self.assertEqual(E.shape, (self.d_e, 4))
        np.testing.assert_array_equal(E[:, 0], E[:, 3])   # same id, same vector
        np.testing.assert_array_equal(E[:, 2], self.W_e[:, -1])

    def test_matches_one_hot_matmul(self):
        v = 4242
        one_hot = np.zeros(self.N_V, dtype=np.float32); one_hot[v] = 1
        np.testing.assert_allclose(token_embedding(v, self.W_e), self.W_e @ one_hot)

    def test_out_of_range(self):
        with self.assertRaises(IndexError): token_embedding(self.N_V, self.W_e)
        with self.assertRaises(IndexError): token_embedding(-1, self.W_e)

    def test_non_integer(self):
        with self.assertRaises(TypeError): token_embedding(1.5, self.W_e)

run_tests(TestTokenEmbedding)

test_matches_one_hot_matmul (__main__.TestTokenEmbedding.test_matches_one_hot_matmul) ... 

ok


test_non_integer (__main__.TestTokenEmbedding.test_non_integer) ... 

ok


test_out_of_range (__main__.TestTokenEmbedding.test_out_of_range) ... 

ok


test_sequence_shape (__main__.TestTokenEmbedding.test_sequence_shape) ... 

ok


test_single_token_shape_and_value (__main__.TestTokenEmbedding.test_single_token_shape_and_value) ... 

ok


----------------------------------------------------------------------
Ran 5 tests in 3.765s

OK


In [4]:
def layer_norm(e, gamma, beta, eps=0.0):
    '''Algorithm 6. e: (d_e,) or (d_e, l). gamma, beta: (d_e,).'''
    e = np.asarray(e, dtype=np.float64)
    m = e.mean(axis=0, keepdims=True)                 # line 1
    v = ((e - m) ** 2).mean(axis=0, keepdims=True)    # line 2
    g, b = (gamma, beta) if e.ndim == 1 else (gamma[:, None], beta[:, None])
    return (e - m) / np.sqrt(v + eps) * g + b         # line 3

In [5]:
class TestLayerNorm(unittest.TestCase):
    def setUp(self):
        self.rng = np.random.default_rng(1)
        self.d = 768
        self.e = self.rng.standard_normal(self.d) * 5 + 3
        self.ones, self.zeros = np.ones(self.d), np.zeros(self.d)

    def test_zero_mean_unit_variance(self):
        out = layer_norm(self.e, self.ones, self.zeros)
        self.assertAlmostEqual(out.mean(), 0.0, places=10)
        self.assertAlmostEqual(out.var(), 1.0, places=10)

    def test_scale_and_offset(self):
        g, b = self.rng.standard_normal(self.d), self.rng.standard_normal(self.d)
        base = layer_norm(self.e, self.ones, self.zeros)
        np.testing.assert_allclose(layer_norm(self.e, g, b), base * g + b)

    def test_explicit_formula(self):
        m = sum(self.e) / self.d
        v = sum((x - m) ** 2 for x in self.e) / self.d
        expected = (self.e - m) / np.sqrt(v)
        np.testing.assert_allclose(layer_norm(self.e, self.ones, self.zeros), expected)

    def test_shift_and_scale_invariance(self):
        a = layer_norm(self.e, self.ones, self.zeros)
        b = layer_norm(4.0 * self.e + 10.0, self.ones, self.zeros)
        np.testing.assert_allclose(a, b, atol=1e-10)

    def test_matrix_is_columnwise(self):
        X = self.rng.standard_normal((self.d, 6))
        g, b = self.rng.standard_normal(self.d), self.rng.standard_normal(self.d)
        out = layer_norm(X, g, b)
        self.assertEqual(out.shape, X.shape)
        for t in range(6):
            np.testing.assert_allclose(out[:, t], layer_norm(X[:, t], g, b))

    def test_eps_prevents_nan_on_constant_input(self):
        const = np.full(self.d, 2.0)
        self.assertTrue(np.isnan(layer_norm(const, self.ones, self.zeros)).any())
        out = layer_norm(const, self.ones, self.zeros, eps=1e-5)
        self.assertTrue(np.all(np.isfinite(out)))

run_tests(TestLayerNorm)

test_eps_prevents_nan_on_constant_input (__main__.TestLayerNorm.test_eps_prevents_nan_on_constant_input) ... 

/tmp/ipykernel_110/3546589150.py:7: RuntimeWarning: invalid value encountered in divide
  return (e - m) / np.sqrt(v + eps) * g + b         # line 3
ok


test_explicit_formula (__main__.TestLayerNorm.test_explicit_formula) ... 

ok


test_matrix_is_columnwise (__main__.TestLayerNorm.test_matrix_is_columnwise) ... 

ok


... (3 outputs left out)

ok


test_zero_mean_unit_variance (__main__.TestLayerNorm.test_zero_mean_unit_variance) ... 

ok


----------------------------------------------------------------------
Ran 6 tests in 0.016s

OK


In [6]:
def softmax_cols(S):
    '''Softmax down each column (over axis 0), numerically stable. Handles -inf entries.'''
    S = S - S.max(axis=0, keepdims=True)
    E = np.exp(S)
    return E / E.sum(axis=0, keepdims=True)

def causal_mask(l_z, l_x=None):
    '''Mask[t_z, t_x] = [[t_z <= t_x]]  -> upper-triangular incl. diagonal, shape (l_z, l_x).'''
    l_x = l_z if l_x is None else l_x
    return np.triu(np.ones((l_z, l_x), dtype=bool))

def attention(X, Z, params, mask=None, return_weights=False):
    '''Algorithm 4.
    X: (d_x, l_x) primary seq; Z: (d_z, l_z) context seq.
    params: dict with Wq (d_attn,d_x), bq (d_attn,), Wk (d_attn,d_z), bk (d_attn,),
            Wv (d_out,d_z), bv (d_out,).
    mask: bool/0-1 array (l_z, l_x) or None (= all ones, i.e. unmasked).
    Returns V_tilde (d_out, l_x) [and the attention weights (l_z, l_x)].'''
    Wq, bq, Wk, bk, Wv, bv = (params[k] for k in ("Wq", "bq", "Wk", "bk", "Wv", "bv"))
    d_attn = Wq.shape[0]
    Q = Wq @ X + bq[:, None]                 # line 1
    K = Wk @ Z + bk[:, None]                 # line 2
    V = Wv @ Z + bv[:, None]                 # line 3
    S = (K.T @ Q) / np.sqrt(d_attn)          # line 4  (l_z, l_x)
    if mask is not None:                     # line 5
        mask = np.asarray(mask).astype(bool)
        if mask.shape != S.shape:
            raise ValueError(f"mask shape {mask.shape} != score shape {S.shape}")
        S = np.where(mask, S, -np.inf)
    A = softmax_cols(S)                      # line 6
    V_tilde = V @ A
    return (V_tilde, A) if return_weights else V_tilde

def init_qkv_params(rng, d_x, d_z, d_attn, d_out, scale=0.02):
    return dict(
        Wq=rng.standard_normal((d_attn, d_x)) * scale, bq=rng.standard_normal(d_attn) * scale,
        Wk=rng.standard_normal((d_attn, d_z)) * scale, bk=rng.standard_normal(d_attn) * scale,
        Wv=rng.standard_normal((d_out, d_z)) * scale,  bv=rng.standard_normal(d_out) * scale,
    )

In [7]:
def algorithm3_reference(e, E_ctx, p):
    '''Algorithm 3 (single query), used as an independent reference. e: (d_in,), E_ctx: (d_in, T).'''
    q = p["Wq"] @ e + p["bq"]
    T = E_ctx.shape[1]
    k = [p["Wk"] @ E_ctx[:, t] + p["bk"] for t in range(T)]
    v = [p["Wv"] @ E_ctx[:, t] + p["bv"] for t in range(T)]
    d = len(q)
    scores = np.array([q @ k[t] / np.sqrt(d) for t in range(T)])
    alpha = np.exp(scores) / np.exp(scores).sum()
    return sum(alpha[t] * v[t] for t in range(T))

class TestAttention(unittest.TestCase):
    def setUp(self):
        self.rng = np.random.default_rng(2)
        self.d_e, self.d_attn, self.l = 768, 64, 8
        self.X = self.rng.standard_normal((self.d_e, self.l))
        self.p = init_qkv_params(self.rng, self.d_e, self.d_e, self.d_attn, self.d_attn, scale=0.05)

    def test_causal_mask_matches_paper_table(self):
        M = causal_mask(8).astype(int)
        self.assertEqual(M.shape, (8, 8))
        np.testing.assert_array_equal(M, np.triu(np.ones((8, 8), dtype=int)))
        self.assertEqual(M[0].sum(), 8); self.assertEqual(M[7].sum(), 1)
        self.assertTrue(all(M[:t+1, t].all() and not M[t+1:, t].any() for t in range(8)))  # column t: t+1 ones then zeros

    def test_output_shape_gpt2(self):
        out = attention(self.X, self.X, self.p, causal_mask(self.l))
        self.assertEqual(out.shape, (64, self.l))

    def test_weights_columns_sum_to_one_and_nonneg(self):
        _, A = attention(self.X, self.X, self.p, causal_mask(self.l), return_weights=True)
        self.assertEqual(A.shape, (self.l, self.l))
        np.testing.assert_allclose(A.sum(axis=0), np.ones(self.l))
        self.assertTrue((A >= 0).all())

    def test_masked_positions_get_exactly_zero_weight(self):
        _, A = attention(self.X, self.X, self.p, causal_mask(self.l), return_weights=True)
        self.assertTrue(np.all(A[np.tril_indices(self.l, k=-1)] == 0.0))   # t_z > t_x

    def test_first_token_attends_only_to_itself(self):
        out = attention(self.X, self.X, self.p, causal_mask(self.l))
        V = self.p["Wv"] @ self.X + self.p["bv"][:, None]
        np.testing.assert_allclose(out[:, 0], V[:, 0])

    def test_causality_future_tokens_do_not_affect_past(self):
        base = attention(self.X, self.X, self.p, causal_mask(self.l))
        X2 = self.X.copy(); X2[:, 5:] = self.rng.standard_normal((self.d_e, self.l - 5))
        new = attention(X2, X2, self.p, causal_mask(self.l))
        np.testing.assert_allclose(base[:, :5], new[:, :5])
        self.assertFalse(np.allclose(base[:, 5:], new[:, 5:]))

    def test_unmasked_is_not_causal(self):
        base = attention(self.X, self.X, self.p, None)
        X2 = self.X.copy(); X2[:, -1] += 1.0
        new = attention(X2, X2, self.p, None)
        self.assertFalse(np.allclose(base[:, 0], new[:, 0]))   # position 1 sees the last token

    def test_zero_queries_give_uniform_average_of_values(self):
        p = dict(self.p); p["Wq"] = np.zeros_like(p["Wq"]); p["bq"] = np.zeros_like(p["bq"])
        out = attention(self.X, self.X, p, causal_mask(self.l))
        V = p["Wv"] @ self.X + p["bv"][:, None]
        for t in range(self.l):
            np.testing.assert_allclose(out[:, t], V[:, :t+1].mean(axis=1))

    def test_matches_algorithm3_per_query(self):
        out = attention(self.X, self.X, self.p, causal_mask(self.l))
        for t in range(self.l):
            ref = algorithm3_reference(self.X[:, t], self.X[:, :t+1], self.p)
            np.testing.assert_allclose(out[:, t], ref, atol=1e-10)

    def test_row_vector_convention_equivalence(self):
        # Standard row-softmax form: softmax(Q K^T / sqrt(d)) V, with causal (lower-tri) mask
        Qr = (self.p["Wq"] @ self.X + self.p["bq"][:, None]).T
        Kr = (self.p["Wk"] @ self.X + self.p["bk"][:, None]).T
        Vr = (self.p["Wv"] @ self.X + self.p["bv"][:, None]).T
        S = Qr @ Kr.T / np.sqrt(self.d_attn)
        S = np.where(np.tril(np.ones((self.l, self.l), bool)), S, -np.inf)
        S = S - S.max(axis=1, keepdims=True)
        A = np.exp(S); A /= A.sum(axis=1, keepdims=True)
        np.testing.assert_allclose(attention(self.X, self.X, self.p, causal_mask(self.l)), (A @ Vr).T, atol=1e-10)

    def test_cross_attention_different_lengths(self):
        Z = self.rng.standard_normal((self.d_e, 11))
        out = attention(self.X, Z, self.p, None)
        self.assertEqual(out.shape, (64, self.l))

    def test_bad_mask_shape(self):
        with self.assertRaises(ValueError):
            attention(self.X, self.X, self.p, np.ones((3, 3)))

    def test_numerical_stability_large_scores(self):
        p = {k: v * 100 for k, v in self.p.items()}
        out = attention(self.X, self.X, p, causal_mask(self.l))
        self.assertTrue(np.all(np.isfinite(out)))

run_tests(TestAttention)

test_bad_mask_shape (__main__.TestAttention.test_bad_mask_shape) ... 

ok


test_causal_mask_matches_paper_table (__main__.TestAttention.test_causal_mask_matches_paper_table) ... 

ok


test_causality_future_tokens_do_not_affect_past (__main__.TestAttention.test_causality_future_tokens_do_not_affect_past) ... 

ok


... (17 outputs left out)

ok

test_zero_queries_give_uniform_average_of_values (__main__.TestAttention.test_zero_queries_give_uniform_average_of_values) ... 

ok


----------------------------------------------------------------------
Ran 13 tests in 0.135s

OK


In [8]:
def mh_attention(X, Z, params, mask=None):
    '''Algorithm 5.
    params: dict with
        'heads': list of H per-head dicts (Wq,bq,Wk,bk,Wv,bv) as in `attention`,
        'Wo': (d_out, H*d_mid), 'bo': (d_out,).
    Returns V_tilde of shape (d_out, l_x).'''
    heads = params["heads"]
    Y = np.concatenate([attention(X, Z, hp, mask) for hp in heads], axis=0)   # lines 1-3
    return params["Wo"] @ Y + params["bo"][:, None]                           # line 4

def init_mha_params(rng, H, d_x, d_z, d_attn, d_mid, d_out, scale=0.02):
    return dict(
        heads=[init_qkv_params(rng, d_x, d_z, d_attn, d_mid, scale) for _ in range(H)],
        Wo=rng.standard_normal((d_out, H * d_mid)) * scale,
        bo=rng.standard_normal(d_out) * scale,
    )

def count_mha_params(params):
    n = sum(a.size for hp in params["heads"] for a in hp.values())
    return n + params["Wo"].size + params["bo"].size

In [9]:
class TestMHAttention(unittest.TestCase):
    def setUp(self):
        self.rng = np.random.default_rng(3)
        self.H, self.d_e, self.d_h, self.l = 12, 768, 64, 8
        self.X = self.rng.standard_normal((self.d_e, self.l))
        self.P = init_mha_params(self.rng, self.H, self.d_e, self.d_e, self.d_h, self.d_h, self.d_e, scale=0.05)
        self.mask = causal_mask(self.l)

    def test_output_shape_gpt2(self):
        self.assertEqual(mh_attention(self.X, self.X, self.P, self.mask).shape, (768, self.l))

    def test_parameter_count_per_layer(self):
        self.assertEqual(count_mha_params(self.P), 2_362_368)

    def test_wo_shape_is_768x768(self):
        self.assertEqual(self.P["Wo"].shape, (768, self.H * self.d_h))
        self.assertEqual(self.H * self.d_h, self.d_e)

    def test_single_head_identity_wo_equals_attention(self):
        rng = np.random.default_rng(4)
        P1 = init_mha_params(rng, 1, self.d_e, self.d_e, 64, 64, 64)
        P1["Wo"], P1["bo"] = np.eye(64), np.zeros(64)
        np.testing.assert_allclose(mh_attention(self.X, self.X, P1, self.mask),
                                   attention(self.X, self.X, P1["heads"][0], self.mask))

    def test_matches_independent_row_convention_implementation(self):
        Xr = self.X.T                                           # (l, d_e)
        outs = []
        for hp in self.P["heads"]:
            Q = Xr @ hp["Wq"].T + hp["bq"]; K = Xr @ hp["Wk"].T + hp["bk"]; V = Xr @ hp["Wv"].T + hp["bv"]
            S = np.einsum("id,jd->ij", Q, K) / np.sqrt(64)
            S = np.where(np.tril(np.ones((self.l, self.l), bool)), S, -np.inf)
            A = np.exp(S - S.max(1, keepdims=True)); A /= A.sum(1, keepdims=True)
            outs.append(A @ V)
        expected = (np.concatenate(outs, axis=1) @ self.P["Wo"].T + self.P["bo"]).T
        np.testing.assert_allclose(mh_attention(self.X, self.X, self.P, self.mask), expected, atol=1e-9)

    def test_causality(self):
        base = mh_attention(self.X, self.X, self.P, self.mask)
        X2 = self.X.copy(); X2[:, 4:] = self.rng.standard_normal((self.d_e, self.l - 4))
        new = mh_attention(X2, X2, self.P, self.mask)
        np.testing.assert_allclose(base[:, :4], new[:, :4])
        self.assertFalse(np.allclose(base[:, 4:], new[:, 4:]))

    def test_heads_use_independent_parameters(self):
        outs = [attention(self.X, self.X, hp, self.mask) for hp in self.P["heads"]]
        self.assertFalse(np.allclose(outs[0], outs[1]))

    def test_head_output_order_in_concat(self):
        # With Wo = identity and bo = 0, output rows [64h:64(h+1)] must equal head h
        P = dict(self.P); P["Wo"], P["bo"] = np.eye(768), np.zeros(768)
        out = mh_attention(self.X, self.X, P, self.mask)
        for h, hp in enumerate(P["heads"]):
            np.testing.assert_allclose(out[64*h:64*(h+1)], attention(self.X, self.X, hp, self.mask))

    def test_cross_attention_lengths(self):
        Z = self.rng.standard_normal((self.d_e, 13))
        self.assertEqual(mh_attention(self.X, Z, self.P, None).shape, (768, self.l))

run_tests(TestMHAttention)

test_causality (__main__.TestMHAttention.test_causality) ... 

ok


test_cross_attention_lengths (__main__.TestMHAttention.test_cross_attention_lengths) ... 

ok


test_head_output_order_in_concat (__main__.TestMHAttention.test_head_output_order_in_concat) ... 

ok

... (9 outputs left out)

ok


test_wo_shape_is_768x768 (__main__.TestMHAttention.test_wo_shape_is_768x768) ... 

ok


----------------------------------------------------------------------
Ran 9 tests in 0.480s

OK


## Demo: one GPT-2 small attention sublayer (LN → causal MHA → residual)

In [10]:
rng = np.random.default_rng(42)
V_, d_e, l = HP['N_V'], HP['d_e'], 8
W_e = rng.standard_normal((d_e, V_)).astype(np.float32) * 0.02
W_p = rng.standard_normal((d_e, HP['l_max'])) * 0.01
ids = np.array([464, 3290, 318, 257, 922, 1517, 286, 262])          # arbitrary 8 token ids

X = token_embedding(ids, W_e) + W_p[:, :l]                          # Alg. 1 + positional embedding
gamma, beta = np.ones(d_e), np.zeros(d_e)
X_tilde = layer_norm(X, gamma, beta, eps=1e-5)                      # Alg. 6 (GPT-2 eps)
P = init_mha_params(rng, HP['H'], d_e, d_e, HP['d_attn'], HP['d_mid'], d_e)
X_out = X + mh_attention(X_tilde, X_tilde, P, causal_mask(l))       # Alg. 5 (-> Alg. 4 x12)
print("X:", X.shape, " X_tilde:", X_tilde.shape, " X_out:", X_out.shape)

X: (768, 8)  X_tilde: (768, 8)  X_out: (768, 8)
